In [59]:
import psycopg2
from sqlalchemy import create_engine
import pandas as pd
import os
import json



In [61]:

# Database connection details
user = 'postgres'
password = '98020927'
host = 'localhost'
port = '5432'
database = 'ADSAC'

# Create the connection string
engine = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}')



In [1]:
class ADSAC_DataBase:
    # Constructor method to initialize attributes
    def __init__(self):
        pass
    

    def get_connection():
        # Database connection details
        host = "localhost"         # e.g., "localhost" or the server IP
        database = "ADSAC" # Name of your database
        user = "postgres"     # Username for PostgreSQL
        password = "98020927" # Password for PostgreSQL

        try:
            # Connect to PostgreSQL database
            connection = psycopg2.connect(
                host=host,
                database=database,
                user=user,
                password=password
            )

            # Create a cursor to interact with the database
            cursor = connection.cursor()

            # Example query
            cursor.execute("SELECT version();")

            # Fetch and print the result of the query
            db_version = cursor.fetchone()
            print("Connected to PostgreSQL database:")
            print(db_version)

        except (Exception, psycopg2.Error) as error:
            print("Error connecting to PostgreSQL:", error)
            cursor = None
        self.cursor = cursor
        self.connection = connection

    def close_connection():
        self.cursor.close()
        self.connection.close()
        print("PostgreSQL connection closed.")

    def insert_values(table_name, data):    
        try:
            # Dynamically create the SQL query
            insert_query = f"INSERT INTO {table_name} VALUES (%s, %s, %s)"
            
            # Execute the query
            self.cursor.execute(insert_query, data)

            # Commit the transaction
            self.connection.commit()

            print("Data inserted successfully!")
            

        except (Exception, psycopg2.Error) as error:
            print("Error while inserting data:", error)
            return False
        return True

    def update_values(table_name, column_to_update, new_value, condition_column, condition_value):    

        try:
            # SQL UPDATE query
            update_query = f"""
            UPDATE {table_name}
            SET {column_to_update} = %s
            WHERE {condition_column} = %s
            """

            # Execute the query with parameters
            self.cursor.execute(update_query, (new_value, condition_value))

            # Commit the transaction
            self.connection.commit()

            print(f"Value updated successfully in {table_name}!")

        except (Exception, psycopg2.Error) as error:
            print("Error while updating values:", error)
            return False
        return True

        
    def delete_values(cursor, table_name, condition_column, condition_value):    
        try:
            # SQL DELETE query
            delete_query = f"""
            DELETE FROM {table_name}
            WHERE {condition_column} = %s
            """

            # Execute the query with parameters
            self.cursor.execute(delete_query, (condition_value,))

            # Commit the transaction
            self.connection.commit()

            print(f"Record deleted successfully from {table_name}!")

        except (Exception, psycopg2.Error) as error:
            print("Error while deleting values:", error)
            return False
        return True        

Connected to PostgreSQL database:
('PostgreSQL 17.4 on x86_64-windows, compiled by msvc-19.42.34436, 64-bit',)
PostgreSQL connection closed.


In [19]:


def upload_sitios_from_directory(path_dir):
    """Carga reviews desde todos los archivos JSON en un directorio a una lista de diccionarios."""
    reviews = []
    for filename in os.listdir(path_dir):
        if filename.endswith(".json"):
            ruta_archivo = os.path.join(path_dir, filename)
            try:
                with open(ruta_archivo, 'r', encoding='utf-8') as f:
                    for line in f:
                        try:
                            review = json.loads(line)
                            reviews.append(review)
                        except json.JSONDecodeError:
                            print(f"Omitiendo línea JSON inválida en {filename}: {line.strip()}")
            except FileNotFoundError:
                print(f"Error: Archivo no encontrado: {ruta_archivo}")
            except json.JSONDecodeError:
                print(f"Error decodificando JSON en el archivo: {ruta_archivo}")
            except Exception as e:
                print(f"Ocurrió un error inesperado al procesar {ruta_archivo}: {e}")
        
        
    return reviews


In [20]:

# Ejemplo de uso:
path_dir = r'C:\Data\metadata-sitios-20250319T175711Z-001'  # Reemplaza con la ruta real a tu directorio
reviews = upload_sitios_from_directory(path_dir)

# Crea un DataFrame de Pandas desde la lista de reviews
df = pd.DataFrame(reviews)

In [54]:
# Imprime el DataFrame para verificar los datos
print(df.size)
df_no_duplicates = df.drop_duplicates(['name','gmap_id','latitude','longitude','address','avg_rating'])
print(df_no_duplicates.size)



28875105
28476420


In [107]:
#Validando los duplicados si aún existen
conteo = df_no_duplicates['gmap_id'].value_counts()

df_conteo = pd.DataFrame(conteo)

indices = df_conteo[df_conteo['count'] > 1].index
print("Índices:", indices)

#print(df_conteo)

Índices: Index([], dtype='object', name='gmap_id')


In [104]:
df_no_duplicates['price'] = df_no_duplicates['price'].apply(lambda x:  0 if x == '$' else x)
df_no_duplicates['price'] = df_no_duplicates['price'].apply(lambda x:  0 if x == '$$' else x)
df_no_duplicates['price'] = df_no_duplicates['price'].apply(lambda x:  0 if x == '$$$' else x)
df_no_duplicates['price'] = df_no_duplicates['price'].apply(lambda x:  0 if x == '$$$$' else x)

df_no_duplicates['price'] = df_no_duplicates['price'].apply(lambda x:  0 if x == '₩' else x)
df_no_duplicates['price'] = df_no_duplicates['price'].apply(lambda x:  0 if x == '₩₩' else x)
df_no_duplicates['price'] = df_no_duplicates['price'].apply(lambda x:  0 if x == '₩₩₩' else x)
df_no_duplicates['price'] = df_no_duplicates['price'].apply(lambda x:  0 if x == '₩₩₩₩' else x)


In [83]:
df_no_duplicates['category'] = df_no_duplicates['category'].astype(str)
df_no_duplicates['hours'] = df_no_duplicates['hours'].astype(str)
df_no_duplicates['MISC'] = df_no_duplicates['MISC'].astype(str)
df_no_duplicates['relative_results'] = df_no_duplicates['relative_results'].astype(str)




C:\Users\DELIA MONTOYA\AppData\Local\Temp\ipykernel_5920\3046549800.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_duplicates['category'] = df_no_duplicates['category'].astype(str)
C:\Users\DELIA MONTOYA\AppData\Local\Temp\ipykernel_5920\3046549800.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_duplicates['hours'] = df_no_duplicates['hours'].astype(str)
C:\Users\DELIA MONTOYA\AppData\Local\Temp\ipykernel_5920\3046549800.py:3: SettingWithCopyWarning: 
A value is trying to be set on a co

In [105]:
# Insert the DataFrame into a PostgreSQL table
table_name = 'sitios'
df_no_duplicates.to_sql(table_name, engine, if_exists='append', index=False)

print(f"Data inserted successfully into the table '{table_name}'!")

Data inserted successfully into the table 'sitios'!
